# 00 - Environment Setup

## CyberForge AI - ML Pipeline Environment Configuration

This notebook sets up the complete environment for the CyberForge AI machine learning pipeline.

### What this notebook does:
1. Validates Python version and system requirements
2. Installs and pins all dependencies
3. Configures GPU/CPU detection
4. Sets up Gemini API connectivity
5. Validates Web Scraper API connection
6. Creates necessary directories

### Prerequisites:
- Python 3.10+ (3.11 recommended)
- Access to Gemini API (API key required)
- Access to WebScrapper.live API

## 1. System Validation

In [ ]:
import sys
import platform
import os
from pathlib import Path

print("=" * 60)
print("CYBERFORGE AI - ENVIRONMENT VALIDATION")
print("=" * 60)

# Python version check
python_version = sys.version_info
print(f"\n✓ Python Version: {python_version.major}.{python_version.minor}.{python_version.micro}")

if python_version.major < 3 or (python_version.major == 3 and python_version.minor < 10):
    raise EnvironmentError("Python 3.10+ is required. Please upgrade your Python installation.")

# System info
print(f"✓ Platform: {platform.system()} {platform.release()}")
print(f"✓ Architecture: {platform.machine()}")
print(f"✓ Processor: {platform.processor() or 'Unknown'}")

# Memory info
try:
    import psutil
    memory = psutil.virtual_memory()
    print(f"✓ Available Memory: {memory.available / (1024**3):.2f} GB / {memory.total / (1024**3):.2f} GB")
except ImportError:
    print("⚠ psutil not installed - memory check skipped")

print("\n" + "=" * 60)

## 2. Install Dependencies

In [ ]:
from pathlib import Path

# Core dependencies with pinned versions for reproducibility
# NOTE: torch/transformers intentionally excluded - not needed for sklearn models
# and too heavy for HF Space Docker containers
DEPENDENCIES = """
# Core ML/AI
numpy>=1.24.0,<2.0.0
pandas>=2.0.0
scikit-learn>=1.3.0
scipy>=1.11.0

# Gemini API (new SDK)
google-genai>=1.0.0

# Data Processing
joblib>=1.3.0
tqdm>=4.65.0
pyarrow>=14.0.0

# Feature Engineering
tldextract>=5.0.0
validators>=0.22.0

# Web/API
httpx>=0.25.0
requests>=2.31.0

# Hugging Face
huggingface_hub>=0.19.0

# Utilities
python-dotenv>=1.0.0
pyyaml>=6.0.0
psutil>=5.9.0
"""

# Write requirements file
requirements_path = Path("../requirements_notebooks.txt")
requirements_path.write_text(DEPENDENCIES.strip())
print(f"✓ Requirements written to: {requirements_path.absolute()}")


In [ ]:
import subprocess
import sys
from pathlib import Path

# Install dependencies
requirements_path = Path("../requirements_notebooks.txt")

if requirements_path.exists():
    print("Installing dependencies... This may take a few minutes.")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_path)],
        capture_output=True,
        text=True
    )

    if result.returncode == 0:
        print("✓ All dependencies installed successfully!")
    else:
        print(f"⚠ Installation warnings: {result.stderr[:500] if result.stderr else 'None'}")
else:
    print("⚠ Requirements file not found. Run previous cell first or skip if deps installed.")


## 3. GPU/CPU Detection

In [ ]:
print("=" * 60)
print("COMPUTE DEVICE DETECTION")
print("=" * 60)

# CyberForge uses sklearn (CPU-only) — torch is optional
try:
    import torch
    cuda_available = torch.cuda.is_available()
    print(f"\n✓ PyTorch Version: {torch.__version__}")
    print(f"✓ CUDA Available: {cuda_available}")

    if cuda_available:
        print(f"✓ CUDA Version: {torch.version.cuda}")
        DEVICE = "cuda"
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        print("✓ Apple MPS (Metal) available")
        DEVICE = "mps"
    else:
        DEVICE = "cpu"
except ImportError:
    print("\n⚠ PyTorch not installed (not required — sklearn models use CPU)")
    DEVICE = "cpu"

print(f"\n✓ Selected Device: {DEVICE}")
print("  (Note: CyberForge models use scikit-learn which runs on CPU)")
print("=" * 60)


## 4. Environment Variables & API Configuration

In [ ]:
import json
import os
from pathlib import Path

# Load configuration from notebook_config.json first (for HF Spaces)
config_json_path = Path("notebook_config.json")
if config_json_path.exists():
    with open(config_json_path, "r") as f:
        loaded_config = json.load(f)
    print(f"✓ Loaded configuration from: {config_json_path.absolute()}")
else:
    loaded_config = {}
    print(f"⚠ No notebook_config.json found, using defaults")

# Try loading .env file as fallback (for local dev)
try:
    from dotenv import load_dotenv
    env_path = Path("../.env")
    if env_path.exists():
        load_dotenv(env_path)
        print(f"✓ Loaded environment from: {env_path.absolute()}")
except ImportError:
    pass

# Detect device (torch is optional)
DEVICE = "cpu"
try:
    import torch
    if torch.cuda.is_available():
        DEVICE = "cuda"
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        DEVICE = "mps"
except ImportError:
    pass

# Configuration class
class Config:
    # API Keys - priority: config.json > env vars > defaults
    GEMINI_API_KEY = loaded_config.get("gemini_api_key") or os.getenv("GEMINI_API_KEY", "")
    HUGGINGFACE_TOKEN = loaded_config.get("hf_token") or os.getenv("HF_TOKEN", "")
    WEBSCRAPER_API_KEY = loaded_config.get("webscraper_api_key", "sk-fd14eaa7bceb478db7afc7256e514d2b")
    WEBSCRAPER_API_URL = loaded_config.get("webscraper_api_url", "http://webscrapper.live/api/scrape")
    
    # Gemini model
    GEMINI_MODEL = loaded_config.get("gemini_model", os.getenv("GEMINI_MODEL", "gemini-2.5-flash"))
    
    # HF repos
    HF_REPO = loaded_config.get("hf_repo", "Che237/cyberforge-models")
    HF_DATASETS_REPO = loaded_config.get("hf_datasets_repo", "Che237/cyberforge-datasets")
    
    # Paths
    BASE_DIR = Path("..").resolve()
    DATASETS_DIR = BASE_DIR / "datasets"
    MODELS_DIR = BASE_DIR / "models"
    ARTIFACTS_DIR = BASE_DIR / "artifacts"
    
    # ML Settings
    RANDOM_STATE = loaded_config.get("random_state", 42)
    TEST_SIZE = loaded_config.get("test_size", 0.2)
    CV_FOLDS = loaded_config.get("cv_folds", 5)
    
    # Device
    DEVICE = DEVICE

config = Config()

# Validate required API keys
print("\n" + "=" * 60)
print("API CONFIGURATION STATUS")
print("=" * 60)
print(f"  Gemini API Key: {'✓ Set' if config.GEMINI_API_KEY else '✗ Missing'}")
hf_status = '✓ Set' if config.HUGGINGFACE_TOKEN else '⚠ Not set (models will not upload)'
print(f"  HuggingFace Token: {hf_status}")
print(f"  Gemini Model: {config.GEMINI_MODEL}")
print(f"  HF Model Repo: {config.HF_REPO}")
print(f"  Device: {config.DEVICE}")


## 5. Gemini API Connectivity Test

In [ ]:
# Gemini Integration — using google-genai (new SDK)
import json
import os
from pathlib import Path

try:
    from google import genai
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'google-genai', '-q'])
    from google import genai

# Load config (self-contained)
config_json_path = Path('notebook_config.json')
if config_json_path.exists():
    with open(config_json_path, 'r') as f:
        loaded_config = json.load(f)
else:
    loaded_config = {}

GEMINI_API_KEY = loaded_config.get('gemini_api_key') or os.getenv('GEMINI_API_KEY', '')
GEMINI_MODEL = loaded_config.get('gemini_model', os.getenv('GEMINI_MODEL', 'gemini-2.5-flash'))

def test_gemini_connection():
    if not GEMINI_API_KEY:
        return False, 'API key not configured'
    try:
        client = genai.Client(api_key=GEMINI_API_KEY)
        response = client.models.generate_content(
            model=GEMINI_MODEL,
            contents='Respond with only: OK'
        )
        return True, f'Model: {GEMINI_MODEL}, Response: {response.text.strip()}'
    except Exception as e:
        # Try fallback model
        try:
            client = genai.Client(api_key=GEMINI_API_KEY)
            response = client.models.generate_content(
                model='gemini-2.5-flash',
                contents='Respond with only: OK'
            )
            return True, f'Model: gemini-2.5-flash (fallback), Response: {response.text.strip()}'
        except Exception as e2:
            return False, str(e2)

print('Testing Gemini API connection...')
success, message = test_gemini_connection()
if success:
    print(f'✓ Gemini API: {message}')
else:
    print(f'⚠ Gemini API: Connection failed - {message}')


## 6. Web Scraper API Connectivity Test

In [ ]:
import httpx
import json
import os
from pathlib import Path

# Load config (self-contained)
config_json_path = Path('notebook_config.json')
if config_json_path.exists():
    with open(config_json_path, 'r') as f:
        loaded_config = json.load(f)
else:
    loaded_config = {}

WEBSCRAPER_API_KEY = loaded_config.get('webscraper_api_key', 'sk-fd14eaa7bceb478db7afc7256e514d2b')
WEBSCRAPER_API_URL = loaded_config.get('webscraper_api_url', 'http://webscrapper.live/api/scrape')

def test_webscraper_connection_sync():
    try:
        with httpx.Client(timeout=30.0) as client:
            response = client.post(
                WEBSCRAPER_API_URL,
                json={'url': 'https://example.com'},
                headers={'Content-Type': 'application/json', 'X-API-Key': WEBSCRAPER_API_KEY}
            )
            if response.status_code == 200:
                return True, 'Connected'
            else:
                return False, f'Status {response.status_code}'
    except Exception as e:
        return False, str(e)

print('Testing Web Scraper API connection...')
success, message = test_webscraper_connection_sync()
if success:
    print(f'✓ WebScraper API: Connected successfully')
else:
    print(f'⚠ WebScraper API: {message}')


## 7. Create Directory Structure

In [ ]:
from pathlib import Path

# Define directories (self-contained)
BASE_DIR = Path('..').resolve()
DATASETS_DIR = BASE_DIR / 'datasets'
MODELS_DIR = BASE_DIR / 'models'
ARTIFACTS_DIR = BASE_DIR / 'artifacts'

# Create necessary directories
directories = [
    DATASETS_DIR,
    MODELS_DIR,
    ARTIFACTS_DIR,
    BASE_DIR / 'logs',
    BASE_DIR / 'cache',
]

print('Creating directory structure...')
for directory in directories:
    directory.mkdir(parents=True, exist_ok=True)
    print(f'  ✓ {directory}')

print('\n✓ Directory structure ready!')


## 8. Save Configuration for Other Notebooks

In [ ]:
import json
import sys
import os
from pathlib import Path

# Get values (self-contained)
python_version = sys.version_info

DEVICE = 'cpu'
torch_version = 'not installed (not required)'
cuda_available = False
try:
    import torch
    torch_version = torch.__version__
    cuda_available = torch.cuda.is_available()
    if cuda_available:
        DEVICE = 'cuda'
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        DEVICE = 'mps'
except ImportError:
    pass

# Load config
config_json_path = Path('notebook_config.json')
if config_json_path.exists():
    with open(config_json_path, 'r') as f:
        loaded_config = json.load(f)
else:
    loaded_config = {}

BASE_DIR = Path('..').resolve()
DATASETS_DIR = BASE_DIR / 'datasets'
MODELS_DIR = BASE_DIR / 'models'
ARTIFACTS_DIR = BASE_DIR / 'artifacts'
RANDOM_STATE = loaded_config.get('random_state', 42)
TEST_SIZE = loaded_config.get('test_size', 0.2)
CV_FOLDS = loaded_config.get('cv_folds', 5)

# Export configuration for other notebooks
notebook_config = {
    'device': str(DEVICE),
    'python_version': f'{python_version.major}.{python_version.minor}.{python_version.micro}',
    'torch_version': torch_version,
    'cuda_available': cuda_available,
    'base_dir': str(BASE_DIR),
    'datasets_dir': str(DATASETS_DIR),
    'models_dir': str(MODELS_DIR),
    'artifacts_dir': str(ARTIFACTS_DIR),
    'random_state': RANDOM_STATE,
    'test_size': TEST_SIZE,
    'cv_folds': CV_FOLDS,
}

config_path = Path('notebook_runtime_config.json')
with open(config_path, 'w') as f:
    json.dump(notebook_config, f, indent=2)

print(f'✓ Configuration exported to: {config_path.absolute()}')
print(json.dumps(notebook_config, indent=2))


## 9. Environment Summary

In [ ]:
import sys
import json
import os
from pathlib import Path

python_version = sys.version_info

try:
    import torch
    torch_version = torch.__version__
    if torch.cuda.is_available():
        DEVICE = 'cuda'
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        DEVICE = 'mps'
    else:
        DEVICE = 'cpu'
except ImportError:
    torch_version = 'not installed'
    DEVICE = 'cpu'

# Load config
config_json_path = Path('notebook_config.json')
if config_json_path.exists():
    with open(config_json_path, 'r') as f:
        loaded_config = json.load(f)
else:
    loaded_config = {}

GEMINI_API_KEY = loaded_config.get('gemini_api_key') or os.getenv('GEMINI_API_KEY', '')
HUGGINGFACE_TOKEN = os.getenv('HF_TOKEN', '')

print('\n' + '=' * 60)
print('ENVIRONMENT SETUP COMPLETE')
print('=' * 60)
print(f'''
✅ Python: {python_version.major}.{python_version.minor}.{python_version.micro}
✅ Device: {DEVICE}
✅ PyTorch: {torch_version}
✅ Gemini API: {'Ready' if GEMINI_API_KEY else 'Not configured'}
✅ HuggingFace: {'Ready' if HUGGINGFACE_TOKEN else 'Using public access'}
✅ WebScraper API: Ready
✅ Directories: Created

You can now proceed to the next notebook:
  → 01_data_acquisition.ipynb
''')
print('=' * 60)
